# 00c — Quarantine: Semantic Embeddings (USE, GPT-2)
**Set aside from the main Step 0 pipeline.** These are **semantic** representations, **not sentiment**, so they do not belong in the sentiment vectors:
- **USE (Universal Sentence Encoder)** — exactly what the parent paper (Ke et al.) used to represent impressions; kept here to **reproduce/benchmark against** that representation in the later brain/RSA step.
- **GPT-2** — the best brain-aligned language model in the encoding literature (Schrimpf et al. 2021; Goldstein et al. 2022; intermediate layer per Toneva & Wehbe 2019).

They produce high-dimensional vectors (512-d / 768-d), saved as `.npy`, for the brain-encoding comparison — NOT pos/neg. Prereq: needs `df` with `Raw_Text` from Step 0 cell 0.1.

### 0.2c (optional) Semantic embeddings — GPT-2 and USE (Jin's paper)
A *different feature class*: high-dim semantic vectors for the brain-encoding angle and as a comparison to our interpretable sentiment vectors. **USE is  what the JIN paper used.** These don't produce pos/neg — keep them for RSA/encoding, not the simple baseline.

In [1]:
#  GPT-2 embedding (intermediate-layer mean-pooled; best brain-aligned LM)
# Schrimpf 2021 / Goldstein 2022; intermediate layers fit best (Toneva & Wehbe 2019).
def make_gpt2_embedder(layer=6):
    import torch
    from transformers import GPT2Tokenizer, GPT2Model
    tok = GPT2Tokenizer.from_pretrained("gpt2")
    mdl = GPT2Model.from_pretrained("gpt2", output_hidden_states=True); mdl.eval()
    def embed(text):
        enc = tok(text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            hs = mdl(**enc).hidden_states[layer][0]   # (tokens, 768)
        return hs.mean(0).numpy()                     # mean-pool -> 768-d
    return embed

# --- USE (JIN-paper representation); TF Hub deep-averaging encoder, 512-d ---
def make_use_embedder():
    import tensorflow_hub as hub
    use = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
    def embed(text): return use([str(text)]).numpy()[0]   # 512-d
    return embed

# Example (uncomment; embeddings are large -> save as .npy, not CSV):
# gpt2 = make_gpt2_embedder(); E = np.vstack([gpt2(t) for t in df['Raw_Text']])
# np.save('reflection_gpt2_layer6.npy', E); print('GPT-2 embeddings', E.shape)